In [1]:
import os
import pandas as pd

In [2]:
def pd_pi_H(pd,pi):
    mean = (pd + pi) / 2
    if mean > 180:
        return mean + 90
    else:
        return mean - 90
    
def pd_pi_V(pd,pi):
    return ((pd - pi) / 2 ) +180

# 1 Reading the File

In [3]:
filepath = 'raw_data.csv'

# Load data
data = pd.read_csv(filepath)

In [4]:
data

,R,E,V,pos,vis,HG,HM,HS,VG,VM,VS,D,hs,hi
0,3,1,2,PD,R,0,0,0,90,59,48,11.508,1.5,1.495
1,3,1,2,PI,R,179,59,56,269,0,9,11.508,1.5,1.495
2,3,1,2,PD,V,199,6,38,88,7,40,13.204,1.5,1.495
3,3,1,2,PI,V,19,6,35,271,52,5,13.204,1.5,1.495
4,1,2,3,PD,R,0,0,0,91,58,56,13.200,1.5,1.533
5,1,2,3,PI,R,179,59,54,268,0,54,13.200,1.5,1.533
6,1,2,3,PD,V,351,5,21,91,33,29,24.358,1.5,1.533
7,1,2,3,PI,V,171,5,19,268,26,14,24.358,1.5,1.533
8,2,3,1,PD,R,0,0,0,88,31,51,24.361,1.5,1.510
9,2,3,1,PI,R,180,0,3,271,28,1,23.361,1.5,1.510


# Degrees (G + M/60 + S/3600)

In [5]:
# computing degrees for H and V components
data['H_degrees'] = data['HG'] + data['HM'] / 60 + data['HS'] / 3600
data['V_degrees'] = data['VG'] + data['VM'] / 60 + data['VS'] / 3600

In [6]:
data

,R,E,V,pos,vis,HG,HM,HS,VG,VM,VS,D,hs,hi,H_degrees,V_degrees
0,3,1,2,PD,R,0,0,0,90,59,48,11.508,1.5,1.495,0.000000,90.996667
1,3,1,2,PI,R,179,59,56,269,0,9,11.508,1.5,1.495,179.998889,269.002500
2,3,1,2,PD,V,199,6,38,88,7,40,13.204,1.5,1.495,199.110556,88.127778
3,3,1,2,PI,V,19,6,35,271,52,5,13.204,1.5,1.495,19.109722,271.868056
4,1,2,3,PD,R,0,0,0,91,58,56,13.200,1.5,1.533,0.000000,91.982222
5,1,2,3,PI,R,179,59,54,268,0,54,13.200,1.5,1.533,179.998333,268.015000
6,1,2,3,PD,V,351,5,21,91,33,29,24.358,1.5,1.533,351.089167,91.558056
7,1,2,3,PI,V,171,5,19,268,26,14,24.358,1.5,1.533,171.088611,268.437222
8,2,3,1,PD,R,0,0,0,88,31,51,24.361,1.5,1.510,0.000000,88.530833
9,2,3,1,PI,R,180,0,3,271,28,1,23.361,1.5,1.510,180.000833,271.466944


In [11]:
# Pairs 

# each one that has same R, E and V and same "vis" shall be put together (mean). But for the case of pd and pi, we need to compute differently, using the functions above.

data_pairs = data.groupby(['R', 'E', 'V', 'vis']).agg({
    "hi" : "mean",
    "D" : "mean",
    'hs' : 'mean',

}).reset_index()

# Compute H_corr and V_corr using the custom functions
data_pairs['H_corr'] = data.groupby(['R', 'E', 'V', 'vis']).apply(lambda x: pd_pi_H(x[x['pos'] == 'PD']['H_degrees'].iloc[0], x[x['pos'] == 'PI']['H_degrees'].iloc[0])).values
data_pairs['V_corr'] = data.groupby(['R', 'E', 'V', 'vis']).apply(lambda x: pd_pi_V(x[x['pos'] == 'PD']['V_degrees'].iloc[0], x[x['pos'] == 'PI']['V_degrees'].iloc[0])).values

import numpy as np

# now horizontal distance (DH) and vertical distance (DV) and height difference (dH)
data_pairs['DH'] = data_pairs['D'] * np.sin(np.radians(data_pairs['V_corr']))
data_pairs['DV'] = data_pairs['D'] * np.cos(np.radians(data_pairs['V_corr']))
data_pairs['dH'] = data_pairs['DV'] + data_pairs['hi'] - data_pairs['hs']

/tmp/ipykernel_226278/716835132.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_pairs['H_corr'] = data.groupby(['R', 'E', 'V', 'vis']).apply(lambda x: pd_pi_H(x[x['pos'] == 'PD']['H_degrees'].iloc[0], x[x['pos'] == 'PI']['H_degrees'].iloc[0])).values
/tmp/ipykernel_226278/716835132.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_pairs['V_corr'] = data.groupby(['R', 'E', 'V', 'vis']).apply(lambda x

In [13]:
data_pairs.to_csv('processed_data.csv', index=False)